# tonight-C — stride sweep (노드 C 담당)

**6 GPU = 3노드 x 2 GPU. 세 노트북은 내용이 동일하고 노드 문자만 다르다.**
잡 정의·우선순위·스킵·집계는 전부 `exp5_tonight.py` 에 있고 이 노트북은 그걸 부르기만 한다.

위에서부터 순서대로: **부팅 → 학습(안 된 것만) → eval(주르륵) → 표.**
셀이 통째로 블로킹되니(학습 ~8h/잡, eval ~2h/셀) 걸어놓고 두면 된다.
중간에 멈춰도 완료분은 skip 되므로 **다시 실행하면 이어서 간다.**

---

## 오늘의 목표 — stride sweep

8/25 K=100 결과 (은지님, n=500, 100k):

| stride | ACT | ACM2 | BiMamba | carry | BiMOS |
|---|---|---|---|---|---|
| 10 | 42.5 | 56.8 | **65.0** | 60.0 | 60.4 |
| 50 | 23.6 | 37.4 | 31.2 | **36.8** | 31.2 |
| 100 | 18.4 | 18.6 | 21.6 | **27.8** | 19.6 |

**전 stride 에서 ACT 를 이기고 s10 에서 +22.5.** 같은 stride = 같은 추론 호출 횟수라
동일 비용 우위다. 그런데 봉우리 양옆(**s5·s15·s25**)이 비어 있어 이게 진짜 봉우리인지,
ACT 가 s5 에서 역전하지 않는지 모른다 — 그걸 채우는 게 1순위.

같은 세팅(K=100 s10)에서 이미 확보된 근거:

| 실험 | 결과 | 의미 |
|---|---|---|
| **스캔 ablation** | plain 56.8 / scan_same(2배 파라미터) 56.0 / BiMamba **65.0** | 파라미터가 아니라 **역방향 스캔** 덕 (+8.2) |
| backbone | r18 65.0 vs r50 51.0 (−14.0) | ResNet18 유지 |
| decoder layer | L1 59.1 / L2 63.4 / L4 48.6 | 2층 |

## 잡 우선순위 (`exp5_tonight._all_jobs`)

1. **K=100 stride sweep** — s5 → s15 → s25 → s10/50/100(완료분 skip) → s75 → s1
   × 6변형 (act / acm2 / bimamba순수 / bimamba_cpoff / bimos / carry)
2. **K sweep @ stride=10 고정** — 메인 그림. 이 축에 지금 K=50(ACT 67.6 / BiMamba 57.8)과
   K=100(42.5 / 65.0) 두 점뿐이라 크로스오버가 50~100 사이라는 것만 알고 곡선이 없다.
   ACT ckpt 는 전 K 에 있어 **재학습 없이 eval 만**으로 채워진다.
3. K=50 stride sweep (ACT 67.6 재현 + carry 표 모순 해소)
4. TE 축 500ep 확정 런

## 노드 분할

`_all_jobs()` / `_all_train_tags()` 는 ckpt 상태를 보지 않는 **고정 목록**이고, 노드는 거기서
`[2::3]` 을 가져간다. 그래서 학습이 끝나 큐가 줄어도 배정이 흔들리지 않고,
**세 노드를 동시에 돌려도 같은 잡을 두 번 돌지 않는다.**


## 0) 부팅

In [ ]:
import sys
from pathlib import Path

_h = Path.cwd()
_r = next(c for c in (_h, *_h.parents) if (c / 'notebooks' / 'libero' / 'exp5_tonight.py').exists())
for _p in (_r / 'notebooks', _r / 'notebooks' / 'libero'):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import importlib
import exp5_tonight as X
X = importlib.reload(X)
cf, v23 = X.setup()          # common_final reload + 태그 등록 (순서 중요)

NODE = 'C'
# 이 노드에서 쓸 GPU. 한 노드에서 창을 2개 띄울 때만 [2, 3] 처럼 직접 지정.
GPUS = v23.available_gpus([0, 1])
print('NODE', NODE, '| GPUS', GPUS)

plan = X.plan(NODE, GPUS)    # 이 노드가 할 일 미리보기 (학습 몇 잡 / eval 몇 셀)


## 1) 학습 체크 → 안 된 것만 학습

`inventory()` 가 30개 태그를 전수 스캔해 `OK / PART / MISS` 로 찍고, `run_trains_all()` 이 **MISS·PART 만** 골라 GPU 수만큼 배치로 돌린다.
이미 150k 인 건 skip, 중단된 건 resume. 이 노드 몫이 다 될 때까지 반복한다.

학습할 게 없으면 `학습할 것 없음` 만 찍고 바로 넘어간다.

In [ ]:
# ── 학습 체크 -> 안 된 것만 학습 ────────────────────────────────────────────
# inventory: 30개 태그(5변형 x K 6개)를 서버에서 전수 스캔. OK / PART(중단) / MISS
# run_trains_all: MISS·PART 만 GPU 수만큼 배치로, 남은 게 없을 때까지. OK 는 자동 skip,
#                 PART 는 resume. 잡당 ~8h 이므로 이 셀은 오래 블로킹된다.
rows = X.inventory()

# ⚠️ 실행 전 확인: bimamba_pure_* 커맨드에 --use_chunk_pairs 가 **없고**
#    --policy.sscp_enabled=false 가 **있어야** 한다. 확인만 하려면 아래 한 줄:
# X.run_trains_all(NODE, GPUS, dry=True)

X.run_trains_all(NODE, GPUS)


## 2) eval 주르륵

이 노드 몫에서 **지금 돌 수 있는 eval 을 전부** 돈다. 완료분은 skip.
다른 노드가 학습 중이라 ckpt 가 없는 셀은 자동으로 빠지고, 그 노드가 끝난 뒤 이 셀을 다시 실행하면 편입돼 이어서 돈다.

In [ ]:
# ── eval 주르륵 ────────────────────────────────────────────────────────────
# 이 노드 몫에서 지금 돌 수 있는 셀을 전부. 완료분은 skip.
# 한 바퀴 끝나면 인벤토리를 다시 읽어서, 그 사이 다른 노드가 학습을 끝냈으면
# 막혀 있던 셀까지 이어서 돈다. 셀당 ~2h.
X.run_evals_all(NODE, GPUS)


## 3) 결과 표

| 결과 | 다음 |
|---|---|
| s5/s15 에서도 BiMamba > ACT | **봉우리가 감싸졌다** → K=100 s10 을 메인 세팅으로 확정 |
| ACT 가 s5 에서 역전 | 봉우리가 왼쪽 → s1~s5 를 더 촘촘히 |
| K sweep 에서 ACT 가 K≥100 부터 무너짐 | 교수님이 요청한 "ACT 가 무너지는 지점" — 메인 그림 확정 |
| 순수 `bimamba` ≈ `bimamba_cpoff` | 오염 무해 → 은지님 기존 값(75.4 등) 그대로 인용 가능 |
| `bimos` < `bimamba` | carry 는 메인에서 빼고 ablation 으로 (K=100 s10 에서 이미 −4.6) |

**주의**: 50ep/task(= overall 500ep) 기준 SE ≈ ±2.2%p. 5%p 미만 차이는 seed/rep 을 늘린 뒤에만 주장할 것.

In [ ]:
# stride sweep -> K sweep -> TE 축 순. 8/25 기존 측정값도 같이 찍어 비교할 수 있게 해둠.
X.report()
